In [1]:
# 一、查询产品及条款
from app.infra.database import AsyncSessionFactory
from app.modules.product.service import ProductService

async with AsyncSessionFactory() as session:
    service = ProductService(session)
    # 查询20条
    # list[Product]
    products= await service.list_products()

# 取一个测试
product = products[0]
print(product)

Product({'id': 1, 'category': 'critical_illness', 'image_url': 'https://files.huizecdn.com/file3/M00/91/12/rBYA3Gi1gZ2AXBL7AAYFJxRhbPg411.png', 'min_premium': Decimal('285.30'), 'target_group': '0-17周岁', 'status': 'active', 'updated_at': '2026-09-09 01:14:26', 'name': '达尔文宝贝计划12号', 'insurer': '信美人寿相互保险社', 'clause_name': '信美相互星辰守护B款少儿重疾险条款.pdf', 'description': '117种重疾+28种中症+45种轻症，少儿重疾保障灵活。', 'max_premium': Decimal('2501.83'), 'highlights': ['117种重疾', '少儿保障', '可选投保人豁免'], 'created_at': '2026-09-09 01:14:26'})


In [2]:
# 二、文档解析
from mineru import MinerU
from app.core.config import settings

# 创建客户端，Precision 模式，需要Token
client = MinerU(settings.rag.mineru_token)
# 解析文件
# ExtractResult
result = client.extract(f"../../data/raw/kb/{product.clause_name}")
result.save_markdown("out/x.md", with_images=True)

WindowsPath('out/x.md')

In [3]:
# 三、文档优化（分块调用，避免单次请求过大被服务端断开）
from pathlib import Path

from langchain.chat_models import init_chat_model

MARKDOWN_OPTIMIZATION_PROMPT = """
下面这份Markdown文档是从保险条款PDF解析得来，由于PDF中的各个小节是以表格形式存在，所以解析时出现错乱。你分析内容，帮我转为格式正确的Markdown，特别是标题编号要正确。
- 标题等级要从1级标题开始，逐层增加，目录和文档名不计入标题等级。
- 输出结果中不要包含正文开始之前的部分。
- 输出结果只包含Markdown正文，不要解释或代码围栏。
- 不要修改保险条款。
""".strip()

llm = init_chat_model(
    settings.llm.chat_model,
    model_provider="deepseek",
    api_key=settings.llm.api_key,
    base_url=settings.llm.base_url,
    extra_body={"thinking": {"type": "disabled"}},
    max_tokens=8000,
)

# 按二级标题切成若干块，逐块交给大模型优化
parts = result.markdown.split("\n## ")
fixed = []
for i, part in enumerate(parts):
    text = part if i == 0 else "## " + part
    if not text.strip():
        continue
    resp = llm.invoke([
        {"role": "system", "content": MARKDOWN_OPTIMIZATION_PROMPT},
        {"role": "user", "content": text},
    ])
    fixed.append(resp.content)
    print(f"[{i + 1}/{len(parts)}] {len(text)} -> {len(resp.content)}")

markdown = "\n\n".join(fixed)

out = Path("out/optimized.md")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(markdown, encoding="utf-8")
print("已保存:", out.resolve(), "| 字符数:", len(markdown))


[1/76] 66 -> 105
[2/76] 312 -> 289
[3/76] 38 -> 5279
[4/76] 1471 -> 1030
[5/76] 4299 -> 4087
[6/76] 366 -> 362
[7/76] 6269 -> 6044
[8/76] 10 -> 8
[9/76] 22 -> 20
[10/76] 6217 -> 5339
[11/76] 705 -> 703
[12/76] 1729 -> 1756
[13/76] 131 -> 121
[14/76] 261 -> 252
[15/76] 152 -> 150
[16/76] 17 -> 15
[17/76] 417 -> 515
[18/76] 2523 -> 2235
[19/76] 103 -> 101
[20/76] 378 -> 376
[21/76] 205 -> 203
[22/76] 350 -> 351
[23/76] 255 -> 253
[24/76] 167 -> 170
[25/76] 79 -> 199
[26/76] 152 -> 145
[27/76] 269 -> 257
[28/76] 219 -> 205
[29/76] 515 -> 511
[30/76] 195 -> 193
[31/76] 2086 -> 1871
[32/76] 194 -> 186
[33/76] 1618 -> 1396
[34/76] 239 -> 232
[35/76] 285 -> 276
[36/76] 107 -> 105
[37/76] 232 -> 237
[38/76] 149 -> 146
[39/76] 665 -> 667
[40/76] 88 -> 83
[41/76] 1761 -> 1522
[42/76] 204 -> 202
[43/76] 360 -> 358
[44/76] 260 -> 259
[45/76] 1842 -> 1628
[46/76] 685 -> 689
[47/76] 1623 -> 1426
[48/76] 334 -> 320
[49/76] 320 -> 318
[50/76] 213 -> 198
[51/76] 156 -> 155
[52/76] 656 -> 652
[53/76] 96

In [4]:
# 四、文档切分
from uuid import uuid4
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from app.rag.models import ParentChunk

# 1.准备TextSpliter
# 1.1.markdown切分器，切父块
markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=[
                ("#", "h1"),
                ("##", "h2"),
                ("###", "h3"),
                ("####", "h4"),
            ],
            strip_headers=True,
        )
# 1.2.递归切分器，切子块
child_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            separators=["\n\n", "\n", "。", "；", ";", "，", ","],
            keep_separator="end",
        )

# 2.先用markdown切分，得到父块小节信息
# list[Document]
sections = markdown_splitter.split_text(markdown)
# 3.处理父子块
parent_chunks: list[ParentChunk] = []
child_chunks: list[Document] = []

# 3.1.遍历父块小节
for section in sections:
    # 3.1.生成章节路径，section.metadata就是每一级标题的字典，values刚好得到标题列表
    section_path = list(section.metadata.values())
    # 3.2.组织 parent_chunk
    parent = ParentChunk(
        id=uuid4(),
        product_id=product.id,
        content=section.page_content,
        section_path=section_path,
        clause_name=product.clause_name
    )
    parent_chunks.append(parent)

    # 3.3.重新组织section的metadata，会作为所有子块的metadata
    section.metadata = {
        "product_id": product.id,
        "parent_id": str(parent.id)
    }
    # 3.4.切分section，生成child的Document列表，并且沿用section的metadata
    children = child_splitter.split_documents([section])
    # 3.5.把父的section_path拼接到子content中，使信息完整
    section_path_header = "\n".join(section_path) + "\n"
    for c in children:
        c.page_content = section_path_header + c.page_content
    child_chunks.extend(children)

[Document(metadata={'h1': '中国平安人寿保险股份有限公司', 'h2': '平安附加智富人生提前给付重大疾病保险条款', 'h3': '阅读指引'}, page_content='请扫描以查询验证条款  \n本阅读指引有助于您理解条款，对本合同内容的解释凡条款已有约定的，以条款约定为准。'), Document(metadata={'h1': '您拥有的重要权益'}, page_content='- 本合同提供的保障在保险责任条款中列明……1.5\n- 在犹豫期内您若要求解除合同，我们向您无息退还保险费……6.1\n- 您有解除合同的权利……6.2'), Document(metadata={'h1': '您应当特别注意的事项'}, page_content='- 本合同设有等待期……1.4\n- 在某些情况下，我们不承担保险责任……2.1\n- 您应当按时交纳保险费……3.1\n- 保险事故发生后，请您及时通知我们……5.2\n- 解除合同会给您造成一定的损失，请您慎重决策……6.2\n- 您有如实告知的义务……7.4\n- 本合同对疾病进行了明确定义，请您仔细阅读……8\n- 我们对一些重要术语进行了解释，并作了显著标识，请您注意……9'), Document(metadata={'h1': '中国太平洋财产保险股份有限公司', 'h2': '总则', 'h3': '第一条'}, page_content='本保险合同（以下简称为“本合同”）由投保单、保险单或其他保险凭证及所附条款，与本合同有关的投保文件、声明、批注、附贴批单及其他书面文件构成。凡涉及本合同的约定，均应采用书面形式。'), Document(metadata={'h1': '中国太平洋财产保险股份有限公司', 'h2': '总则', 'h3': '第二条'}, page_content='凡在中华人民共和国境内（不包括港澳台地区）依法设立并登记注册，从事生产经营活动的企业，均可作为本合同的投保人、被保险人。'), Document(metadata={'h1': '中国太平洋财产保险股份有限公司', 'h2': '保险责任', 'h3': '第三条'}, page_content='在保险期间内，被保险人在本保险合同载明的场所范围内从事生产、经营等活动过程中，因发生生产

In [ ]:
# 五、存入向量库
from langchain_ollama import OllamaEmbeddings
from langchain_milvus import Milvus, BM25BuiltInFunction
from app.core.config import settings

# 1.初始化向量模型
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

# 2、初始化 VectorStore，会自动创建Collection
vector_store = Milvus(
    embedding_function=embeddings,
    collection_name="insurance_collection",
    builtin_function=BM25BuiltInFunction(analyzer_params={"type": "chinese"}),
    vector_field=["dense", "sparse"],
    connection_args={"uri": settings.rag.milvus_url},
    auto_id=True,
)

# 3.删除旧文档
vector_store.delete(expr=f"product_id=={product.id}")
# 4.添加新文档
vector_store.add_documents(child_chunks)

In [ ]:
# 六、父块入数据库
from app.infra.database import AsyncSessionFactory
from app.rag.repository import ParentChunkRepository

async with AsyncSessionFactory() as session:
    async with session.begin():
        repository = ParentChunkRepository(session)
        # 先删除旧数据
        await repository.delete_by_product_id(product.id)
        # 再添加新数据
        repository.add_all(parent_chunks)